**Table of contents**<a id='toc0_'></a>    
- [Photoswitching fingerprints: 2 fluorophores](#toc1_)    
  - [Data generation](#toc1_1_)    
  - [Reading and fitting the data](#toc1_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Photoswitching fingerprints: 2 fluorophores](#toc0_)

In [18]:
import glob
import numpy as np
import pandas as pd

import fluopy.distributions as dist
import fluopy.figure as fi
import fluopy.fitting as fit
import fluopy.fluorophores as fl
import fluopy.transitions as tr
import fluopy.routines as rt

%load_ext autoreload
%autoreload 2

import warnings


def custom_warning_format(msg, category, filename, lineno, line=None):
    if line is None:
        import linecache

        line = linecache.getline(filename, lineno)
    return f"WARNING for line: {line} {msg} \n"


warnings.formatwarning = custom_warning_format

for package in [dist, fi, fl, tr, rt]:
    print(f"{package.__name__} version: {package.__version__}")

saving_at = r"D:\python_output\Chapter_I\0_4_multi_f_PFA\2F"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
fluopy.distributions version: 0.1.0
fluopy.figure version: 0.1.0
fluopy.fluorophores version: 0.1.0
fluopy.transitions version: 0.1.0
fluopy.routines version: 0.1.0


## <a id='toc1_1_'></a>[Data generation](#toc0_)

In [ ]:
rng = np.random.default_rng(1)


def prepare_transition_set(distance):
    fluorophores = fl.construct_fluorophores(name="cy5_dna", count=2, distance=distance)
    fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)
    transitions = fluorophore_system.load_transitions(
        bleaching=True,
        summarize=True,
        energy_transfer=True,
        **rt.PARAMS_DSTORM,
    )
    transition_set = tr.TransitionSet(transitions, fluorophore_system)
    transition_set.finalize()

    return transition_set

In [ ]:
# 900 min
distances = [3, 6, 9, 18]
for distance in distances:
    transition_set = prepare_transition_set(distance)
    _, _, _ = rt.fingerprint_analysis(
        transition_set,
        batch_size=80,
        batches=1,
        filepath=saving_at,
        filename=f"{distance}nm",
        seed=1,
        use_memmap=r"C:\Users\vie43sq\Desktop\memmaps\run_1",
    )

## <a id='toc1_2_'></a>[Reading and fitting the data](#toc0_)

In [ ]:
distances = [3, 6, 9, 18]
identifiers = [f"{distance}nm" for distance in distances]
fingerprints_all = []
x = np.linspace(0, 300, 300001)
for i, id in enumerate(identifiers):
    fingerprints_all.append(
        pd.Series(np.zeros(300001), np.round(x, decimals=12), dtype=np.int32)
    )
    for file in glob.glob(saving_at + "/*"):
        if file.endswith(".parquet") and id in file:
            fingerprints_all[i] += pd.read_parquet(file).sum(axis=1)
    fingerprint = fingerprints_all[i].cumsum() / fingerprints_all[i].sum()
    y = fingerprint.values
    result = fit.ps_fingerprint_cdf_fit_2f(x, y, maxiter=1000, popsize=50, disp=False)
    np.save(saving_at + "\\" + rf"fitting_parameters\\{id}.npy", result.x)